# 09 - Final inference and evaluation: every system, same questions, same decoding

> Run with **Save Version -> Save & Run All (Commit)**. Settings: **Accelerator = GPU T4 x2**, **Internet = On**.
> This is the **inference notebook** of the submission: it loads the trained LoRA checkpoints, answers the hold-out test questions and the
> 1,000 unseen questions (Xt), and scores every system with the shared harness. No training.

**Inputs (Add Input):**
1. `train_processed.csv` (dataset `train-processed1`) and `test_processed.csv` (dataset `train-processed`: the unseen 1,000 questions)
2. `question_neighbours.json` (dataset `nlp-qna-retrieval`: original retrieval order)
3. `question_neighbours_reranked.json.gz` (Your Work -> re-ranker notebook 07 output)
4. the Phase 4 adapters (dataset `nlp-qna-adapters`: `nn/`, `none/`)
5. the retrained adapter (Your Work -> retrain notebook 08 output: `adapters/nn/final`)

**Systems on the hold-out test questions** (the same seed-42 sample of 300 as Phase 5; questions never seen in training):

| system | model | reference answers in the prompt |
|---|---|---|
| B0a (original) | none - copy the nearest training answer | - |
| B0a (re-ranked) | none - copy the top re-ranked training answer | - |
| B1 | Qwen2.5-3B-Instruct zero-shot | none |
| B2 | Phase 4 adapter `none` | none |
| P1 | Phase 4 adapter `nn` (4,000 rows) | top-2, original order |
| **P2 (final)** | retrained adapter (all 12,825 rows) | **top-2, re-ranked** |
| P2 ablation | retrained adapter | top-2, original order |

All generated systems use the **same decoding**, chosen in the decoding sweep (notebook 06): greedy, repetition penalty 1.1 on the answer's own tokens,
no repeated 12-token n-gram inside the answer, at most 1,280 new tokens.

**Unseen set (Xt):** exact repeats of a training question -> stored medoid answer (router); every other question -> P2.
Writes `xt_predictions.csv`, `xt_predictions_detailed.csv`, `xt_predictions_with_baseline.csv` and `metrics.json`.

In [ ]:
!pip install -q -U peft bitsandbytes accelerate huggingface_hub rouge-score sacrebleu

In [ ]:
%%writefile prompts.py
"""Phase 4: build the training / inference prompts.

Modes
  none : question only                                   (baseline B2: plain fine-tuning)
  nn   : question + the 2 nearest training Q&A pairs      (proposed model without PDFs)
  full : nn + source-page excerpts                         (proposed model with PDFs)

Neighbours for TRAIN questions already exclude the question's own group (see Phase 2), so the model is trained to
synthesise an answer from related-but-different answers, not to copy an identical one.
Pure ASCII source on purpose.
"""
import json
import random

SYSTEM = (
    "You are an expert Indian agricultural advisor. Answer the farmer's question in English with a detailed, "
    "well-structured advisory: key facts or symptoms, thresholds where relevant, cultural, biological and chemical "
    "management with exact doses and units, and safety notes. Use the reference material when it applies to this "
    "question; never invent doses."
)


def trim_words(text, n):
    w = text.split()
    return " ".join(w[:n]) + (" ..." if len(w) > n else "")


def trim_tokens(text, n, tok):
    """Cut text to n tokens. Token budgets (not word counts) keep every prompt inside the length limit: numeric-heavy
    text can use twice as many tokens per word as prose."""
    ids = tok(text, add_special_tokens=False)["input_ids"]
    return text if len(ids) <= n else tok.decode(ids[:n]) + " ..."


def build_user(mode, question, refs, excerpts, ref_words=300, pdf_words=180, tok=None, ref_tokens=380, pdf_tokens=200):
    """refs: list of (question, answer); excerpts: list of (label, text). With tok given, trimming is by tokens."""
    def cut(text, words, tokens):
        return trim_tokens(text, tokens, tok) if tok is not None else trim_words(text, words)
    if mode == "none":
        return "Question: " + question
    parts = ["Reference answers to similar questions:"]
    for i, (q, a) in enumerate(refs, 1):
        parts.append("[%d] Q: %s\nA: %s" % (i, q, cut(a, ref_words, ref_tokens)))
    if mode == "full" and excerpts:
        parts.append("Source excerpts:")
        for label, t in excerpts:
            parts.append("[%s] %s" % (label, cut(t, pdf_words, pdf_tokens)))
    parts.append("Question: " + question)
    return "\n\n".join(parts)


class Context:
    """Holds the lookups needed to build prompts for any query set."""

    def __init__(self, rows, page_text=None, chunk_text=None):
        self.qid2row = {r["question_id"]: r for r in rows}
        self.rows = rows
        self.page_text = page_text or {}      # (doc_id, page) -> first chunk text of that page
        self.chunk_text = chunk_text or []    # chunk id -> text
        self._cit = {}

    def citations(self, qid):
        if qid not in self._cit:
            self._cit[qid] = json.loads(self.qid2row[qid]["citations_json"])
        return self._cit[qid]

    def propagated_pages(self, neighbour_ids, limit=2):
        """Pages cited by the nearest training questions, in neighbour order. Returns [(label, text)]."""
        out, seen = [], set()
        for nid in neighbour_ids:
            for c in self.citations(nid):
                if c.get("kind") != "zoho":
                    continue
                for p in c.get("pages", []):
                    key = (c["doc_id"], p)
                    if key in self.page_text and key not in seen:
                        seen.add(key)
                        name = (c.get("name") or "source")[:60]
                        out.append(("%s, p.%d" % (name, p), self.page_text[key]))
                        if len(out) == limit:
                            return out
        return out


def make_example(ctx, mode, qid, question, neighbours, chunk_hits=None, answer=None, noise=0.0, rng=None,
                 pool_qids=None, n_refs=2, tok=None):
    """neighbours: [[question_id, score], ...] best first."""
    rng = rng or random.Random(0)
    nids = [n[0] for n in neighbours]
    refs_ids = nids[:n_refs]
    if noise and pool_qids and len(refs_ids) == n_refs and rng.random() < noise:
        refs_ids[-1] = rng.choice(pool_qids)      # RAFT-style distractor: the model learns to ignore bad context
    refs = [(ctx.qid2row[i]["question"], ctx.qid2row[i]["final_answer"]) for i in refs_ids]
    excerpts = []
    if mode == "full":
        excerpts = ctx.propagated_pages(nids[:3])
        if len(excerpts) < 2 and chunk_hits and ctx.chunk_text:
            for cid in chunk_hits:
                if len(excerpts) == 2:
                    break
                excerpts.append(("retrieved passage", ctx.chunk_text[cid]))
        if noise and excerpts and pool_qids and rng.random() < noise:
            other = ctx.propagated_pages([rng.choice(pool_qids)], limit=1)
            if other:
                excerpts[-1] = other[0]
    ex = {"qid": qid, "mode": mode, "system": SYSTEM, "user": build_user(mode, question, refs, excerpts, tok=tok)}
    if answer is not None:
        ex["answer"] = answer
    return ex

In [ ]:
%%writefile eval_harness.py
"""Single evaluation harness used by every system (baselines, ablations, final model).

    from eval_harness import evaluate
    scores = evaluate(preds, refs)            # dicts {question_id: text}
    scores = evaluate(preds, refs, bertscore=True, meteor=True)   # slower extras

Metrics: ROUGE-1/2/L F1, corpus BLEU (sacrebleu), length ratio, agronomic fact recall/precision/F1
(number+unit pairs, formulation-tagged chemicals such as "dinotefuran 20 SG", binomials), and optionally
METEOR (nltk) and BERTScore-F1. Extras are lazy-imported so the core runs with rouge-score + sacrebleu only.
"""
import re
import statistics as st

from rouge_score import rouge_scorer

_UNIT = {"gm": "g", "gram": "g", "grams": "g", "gms": "g", "litre": "l", "litres": "l", "lit": "l", "liter": "l",
         "liters": "l", "hectare": "ha", "hectares": "ha", "per cent": "%", "percent": "%", "quintals": "q",
         "quintal": "q", "tonnes": "t", "tonne": "t", "day": "days", "kgs": "kg"}
NUM_UNIT_RE = re.compile(
    r"(\d+(?:\.\d+)?)\s*(?:-|\u2013|to)?\s*(\d+(?:\.\d+)?)?\s*"
    r"(kg|kgs|gms?|grams?|g|mg|ml|litres?|liters?|lit|l|quintals?|q|tonnes?|t|ha|hectares?|acre|cm|mm|"
    r"days?|per\s*cent|percent|%|ppm)\b"
    r"(?:\s*(?:/|per)\s*(ha|hectare|acre|litres?|liters?|lit|l|plant|kg|m2|10\s*l))?", re.I)
CHEM_RE = re.compile(r"\b([A-Za-z][A-Za-z\-]{3,})\s+\d+(?:\.\d+)?\s*%?\s*"
                     r"(SG|SL|EC|WP|WG|SC|FS|DP|GR|ZC|OD|EW|CS|ME|WS|DF)\b")
BINOM_RE = re.compile(r"\(([A-Z][a-z]{3,}\s+[a-z]{4,})\)")


def _u(x):
    x = re.sub(r"\s+", " ", x.lower().strip())
    return _UNIT.get(x, x)


def extract_facts(text):
    facts = set()
    for m in NUM_UNIT_RE.finditer(text):
        a, b, unit, per = m.groups()
        facts.add("%s%s%s%s" % (a, "-" + b if b else "", _u(unit), "/" + _u(per) if per else ""))
    for m in CHEM_RE.finditer(text):
        facts.add("chem:" + m.group(1).lower())
    for m in BINOM_RE.finditer(text):
        facts.add("sp:" + m.group(1).lower())
    return facts


def _fact_scores(preds, refs):
    r_list, p_list, f_list = [], [], []
    for p, r in zip(preds, refs):
        fr, fp = extract_facts(r), extract_facts(p)
        if not fr:
            continue
        inter = len(fr & fp)
        rec = inter / len(fr)
        prec = inter / len(fp) if fp else 0.0
        r_list.append(rec)
        p_list.append(prec)
        f_list.append(0.0 if rec + prec == 0 else 2 * rec * prec / (rec + prec))
    n = max(1, len(r_list))
    return {"fact_recall": sum(r_list) / n, "fact_precision": sum(p_list) / n, "fact_f1": sum(f_list) / n,
            "fact_n_refs": len(r_list)}


def evaluate(preds, refs, bertscore=False, meteor=False, bert_model="roberta-large"):
    ids = [i for i in refs if i in preds]
    P = [preds[i] or "" for i in ids]
    R = [refs[i] for i in ids]
    sc = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    agg = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    for p, r in zip(P, R):
        s = sc.score(r, p)
        for k in agg:
            agg[k] += s[k].fmeasure
    n = max(1, len(ids))
    out = {"n": len(ids), "n_missing": len(refs) - len(ids)}
    out.update({k: v / n for k, v in agg.items()})
    import sacrebleu
    out["bleu"] = sacrebleu.corpus_bleu(P, [R]).score / 100.0
    pl, rl = [len(p.split()) for p in P], [len(r.split()) for r in R]
    out["len_ratio_mean"] = (sum(pl) / n) / max(1e-9, sum(rl) / n)
    out["len_ratio_median_per_item"] = st.median([a / max(1, b) for a, b in zip(pl, rl)]) if ids else 0.0
    out.update(_fact_scores(P, R))
    if meteor:
        from nltk.translate.meteor_score import meteor_score
        out["meteor"] = sum(meteor_score([r.split()], p.split()) for p, r in zip(P, R)) / n
    if bertscore:
        from bert_score import score as bs
        _, _, f1 = bs(P, R, model_type=bert_model, lang="en", batch_size=16, verbose=False)
        out["bertscore_f1"] = float(f1.mean())
    return out


def evaluate_by_group(preds, refs, groups, **kw):
    """groups: {name: set(question_ids)} -> {name: scores}; e.g. novel vs repeated, or per crop."""
    return {g: evaluate(preds, {i: refs[i] for i in ids if i in refs}, **kw) for g, ids in groups.items()}


if __name__ == "__main__":
    t = "Spray dinotefuran 20 SG @ 60 g/acre; seed rate 25 kg/ha. Pest: Amrasca devastans (Amrasca devastans)."
    print(extract_facts(t))
    print(evaluate({"a": t}, {"a": t}))

In [ ]:
%%writefile infer.py
"""Phase 5 generation worker: one process per GPU.

Reads a jobs file (jsonl: uid, adapter, prio, system, user), generates greedy answers in batches, and appends every
finished batch to --out immediately, so a crashed or time-limited run resumes where it stopped (done uids are skipped).
adapter is "nn" / "none" (a LoRA folder under --adapter_dir) or "base" (the same model with the adapter switched off:
the zero-shot baseline B1).  Pure ASCII source on purpose.

    CUDA_VISIBLE_DEVICES=0 python infer.py --jobs jobs.jsonl --adapter_dir /path --adapters nn --out gen_gpu0.jsonl
"""
import argparse
import json
import os
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, LogitsProcessor, LogitsProcessorList
from peft import PeftModel


def parse():
    p = argparse.ArgumentParser()
    p.add_argument("--jobs", required=True)
    p.add_argument("--adapter_dir", required=True)
    p.add_argument("--adapters", required=True, help="comma list handled by this worker, e.g. none,base")
    p.add_argument("--out", required=True)
    p.add_argument("--model", default="Qwen/Qwen2.5-3B-Instruct")
    p.add_argument("--max_new", type=int, default=768)
    p.add_argument("--batch", type=int, default=12)
    p.add_argument("--rep_penalty", type=float, default=1.05, help="HF repetition penalty (acts on prompt AND answer tokens)")
    p.add_argument("--gen_rep_penalty", type=float, default=1.0, help="penalty on tokens of the ANSWER only (1.0 = off)")
    p.add_argument("--gen_no_repeat", type=int, default=0, help="forbid repeating any n-gram of the ANSWER (0 = off)")
    p.add_argument("--do_sample", action="store_true")
    p.add_argument("--temperature", type=float, default=0.7)
    p.add_argument("--top_p", type=float, default=0.9)
    p.add_argument("--seed", type=int, default=1234)
    p.add_argument("--hours", type=float, default=10.0)
    p.add_argument("--no_quant", action="store_true")
    return p.parse_args()


class GenRepetitionPenalty(LogitsProcessor):
    """Repetition penalty over the generated tokens only (the HF one also penalises every prompt token, which
    discourages copying wording from the reference answers in the prompt). Assumes left padding."""

    def __init__(self, penalty, prompt_len):
        self.p, self.plen = penalty, prompt_len

    def __call__(self, input_ids, scores):
        g = input_ids[:, self.plen:]
        if g.shape[1] == 0:
            return scores
        s = torch.gather(scores, 1, g)
        s = torch.where(s < 0, s * self.p, s / self.p)
        return scores.scatter(1, g, s)


class GenNoRepeatNGram(LogitsProcessor):
    """Forbid any n-gram that already occurs in the generated answer (prompt excluded). Incremental per row."""

    def __init__(self, n, prompt_len):
        self.n, self.plen, self.seen = n, prompt_len, None

    def __call__(self, input_ids, scores):
        B, L = input_ids.shape
        if self.seen is None:
            self.seen = [dict() for _ in range(B)]
        glen = L - self.plen
        if glen < self.n - 1:
            return scores
        rows = input_ids[:, self.plen:].tolist()
        for b in range(B):
            g = rows[b]
            if glen >= self.n:                      # register the newest complete n-gram
                self.seen[b].setdefault(tuple(g[-self.n:-1]), set()).add(g[-1])
            banned = self.seen[b].get(tuple(g[len(g) - self.n + 1:]))
            if banned:
                scores[b, list(banned)] = -float("inf")
        return scores


def load_model(a, names):
    cuda = torch.cuda.is_available()
    if a.no_quant:
        base = AutoModelForCausalLM.from_pretrained(a.model, torch_dtype=torch.float16 if cuda else torch.float32)
        if cuda:
            base = base.to("cuda:0")
    else:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                                 bnb_4bit_use_double_quant=True)
        base = AutoModelForCausalLM.from_pretrained(a.model, quantization_config=bnb, torch_dtype=torch.float16,
                                                    device_map={"": 0})
    real = [n for n in names if n != "base"]
    if not real:
        return base.eval(), False
    model = PeftModel.from_pretrained(base, os.path.join(a.adapter_dir, real[0]), adapter_name=real[0])
    for n in real[1:]:
        model.load_adapter(os.path.join(a.adapter_dir, n), adapter_name=n)
    return model.eval(), True


def generate(model, tok, texts, a, device):
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(device)
    plen = enc["input_ids"].shape[1]
    procs = LogitsProcessorList()
    if getattr(a, "gen_rep_penalty", 1.0) != 1.0:
        procs.append(GenRepetitionPenalty(a.gen_rep_penalty, plen))
    if getattr(a, "gen_no_repeat", 0):
        procs.append(GenNoRepeatNGram(a.gen_no_repeat, plen))
    kw = dict(max_new_tokens=a.max_new, repetition_penalty=a.rep_penalty, pad_token_id=tok.pad_token_id,
              logits_processor=procs)
    if getattr(a, "do_sample", False):
        torch.manual_seed(a.seed)
        kw.update(do_sample=True, temperature=a.temperature, top_p=a.top_p, top_k=0)
    else:
        kw.update(do_sample=False, temperature=None, top_p=None, top_k=None)
    with torch.no_grad():
        out = model.generate(**enc, **kw)
    gen = out[:, enc["input_ids"].shape[1]:]
    res = []
    for g in gen:
        ids = g.tolist()
        n = len(ids)
        while n and ids[n - 1] == tok.pad_token_id:
            n -= 1
        res.append((tok.decode(ids[:n], skip_special_tokens=True).strip(), n))
    return res


def main():
    a = parse()
    names = a.adapters.split(",")
    jobs = [json.loads(l) for l in open(a.jobs, encoding="utf-8")]
    jobs = [j for j in jobs if j["adapter"] in names]
    done = set()
    if os.path.exists(a.out):
        done = {json.loads(l)["uid"] for l in open(a.out, encoding="utf-8") if l.strip()}
    todo = [j for j in jobs if j["uid"] not in done]
    print("jobs for %s: %d | already done: %d | to do: %d" % (names, len(jobs), len(done), len(todo)), flush=True)
    if not todo:
        return
    tok = AutoTokenizer.from_pretrained(a.model)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    for j in todo:
        j["text"] = tok.apply_chat_template([{"role": "system", "content": j["system"]}, {"role": "user", "content": j["user"]}],
                                            tokenize=False, add_generation_prompt=True)
        j["plen"] = len(tok(j["text"], add_special_tokens=False)["input_ids"])
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model, has_peft = load_model(a, names)
    t_end = time.time() + a.hours * 3600
    t0 = time.time()
    n_done = 0
    fout = open(a.out, "a", encoding="utf-8")
    for name in names:
        group = sorted((j for j in todo if j["adapter"] == name), key=lambda j: (j["prio"], -j["plen"]))
        if not group:
            continue
        if name != "base" and has_peft:
            model.set_adapter(name)
        print("adapter %s: %d jobs" % (name, len(group)), flush=True)
        i = 0
        bs = a.batch
        while i < len(group):
            if time.time() > t_end:
                print("time limit reached - stopping; rerun to resume", flush=True)
                return
            batch = group[i:i + bs]
            try:
                if name == "base" and has_peft:
                    with model.disable_adapter():
                        res = generate(model, tok, [j["text"] for j in batch], a, device)
                else:
                    res = generate(model, tok, [j["text"] for j in batch], a, device)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs == 1:
                    raise
                bs = max(1, bs // 2)
                print("out of memory - batch size now", bs, flush=True)
                continue
            for j, (text, n) in zip(batch, res):
                fout.write(json.dumps({"uid": j["uid"], "adapter": name, "text": text, "n_tokens": n,
                                       "hit_limit": n >= a.max_new}, ensure_ascii=False) + "\n")
            fout.flush()
            i += len(batch)
            n_done += len(batch)
            el = time.time() - t0
            print("[%5.0f min] %s %d/%d | %.1f s per answer" % (el / 60, name, i, len(group), el / n_done), flush=True)
    print("DONE", n_done, "answers in %.1f min" % ((time.time() - t0) / 60), flush=True)


if __name__ == "__main__":
    main()

## 0. Configuration and inputs

In [ ]:
import glob, os, sys, json, gzip, csv, random, time, shutil, subprocess, collections as C
csv.field_size_limit(10 ** 9)
_missing = [f for f in ("prompts.py", "eval_harness.py", "infer.py") if not os.path.exists(f)]
assert not _missing, "missing " + str(_missing) + " - use Run All from the top"
sys.path.insert(0, os.getcwd())
import torch
CPU = bool(os.environ.get("P9_CPU"))      # CPU smoke-test hook, never set on Kaggle
if not CPU:
    assert torch.cuda.is_available(), "No GPU. Session options -> Accelerator -> GPU T4 x2, then Run All."
N_GPU = torch.cuda.device_count()
print("GPUs:", N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])

MODEL = os.environ.get("P9_MODEL", "Qwen/Qwen2.5-3B-Instruct")
N_HOLDOUT = int(os.environ.get("P9_NHOLD", "300"))   # same seed-42 sample as Phase 5; 0 = all 1,500 (about 5x slower)
ABLATION_ORIG_ORDER = True     # also run the final model with the ORIGINAL retrieval order (isolates the re-ranker at inference)
DECODE = {"max_new": int(os.environ.get("P9_MAXNEW", "1280")), "rep_penalty": 1.0, "gen_rep_penalty": 1.1, "gen_no_repeat": 12}
XT_LIMIT = int(os.environ.get("P9_XT_LIMIT", "0"))  # smoke tests only (0 = all)
BATCH = 10
GEN_HOURS = 10.5               # each worker stops cleanly after this; re-run to resume (finished answers are kept)
BERTSCORE = False
WORK = os.environ.get("P9_WORK", "/kaggle/working")
INPUT = os.environ.get("P9_INPUT", "/kaggle/input")

def find(pattern, required=True):
    hit = sorted(p.replace("\\", "/") for p in glob.glob(INPUT + "/**/" + pattern, recursive=True))
    if not hit and required:
        raise AssertionError("attach a dataset / notebook output containing " + pattern)
    return hit[0] if hit else None

def op(path):
    return gzip.open(path, "rt", encoding="utf-8") if path.endswith(".gz") else open(path, encoding="utf-8")

TRAIN_CSV, UNSEEN_CSV = find("train_processed*.csv"), find("test_processed*.csv")
NN_ORIG, NN_RR = find("question_neighbours.json*"), find("question_neighbours_reranked.json*")
V1_NN = os.path.dirname(find("nn/adapter_config.json"))
V1_NONE = os.path.dirname(find("none/adapter_config.json"))
V2_NN = os.path.dirname(find("adapters/nn/final/adapter_config.json"))
V2_SUMMARY = os.path.join(os.path.dirname(V2_NN), "train_summary.json")
V2_STATUS = json.load(open(V2_SUMMARY)) if os.path.exists(V2_SUMMARY) else {}
print("retrained adapter:", V2_NN, "| training status:", V2_STATUS.get("status"), "| steps:", V2_STATUS.get("steps"),
      "of", V2_STATUS.get("steps_per_epoch"))
if V2_STATUS.get("status") != "completed":
    print("WARNING: the retrained adapter did not finish its epoch - results will describe a PARTIALLY trained model.")

AD = WORK + "/ad"                                  # one folder with all three adapters under distinct names
for name, srcdir in (("nn_v1", V1_NN), ("none_v1", V1_NONE), ("nn_v2", V2_NN)):
    if not os.path.exists(AD + "/" + name):
        shutil.copytree(srcdir, AD + "/" + name)
print("adapters:", {n: len(os.listdir(AD + "/" + n)) for n in os.listdir(AD)}, "| decoding:", DECODE, "| hold-out N:", N_HOLDOUT)

## 1. Router and prompts (CPU)

In [ ]:
import prompts as P
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)
rows = list(csv.DictReader(open(TRAIN_CSV, encoding="utf-8")))
unseen = list(csv.DictReader(open(UNSEEN_CSV, encoding="utf-8")))
NNO, NNR = json.load(op(NN_ORIG)), json.load(op(NN_RR))
ctx = P.Context(rows)
qid2row = ctx.qid2row

# exact-match router: unseen question text == a training question -> that group's medoid answer
medoid = {r["group_id"]: r for r in rows if r["is_medoid"] == "1"}
routed = {u["question_id"]: medoid[u["matched_group_id"]]["final_answer"]
          for u in unseen if u["in_train"] == "1" and u["matched_group_id"] in medoid}
gen_unseen = [u for u in unseen if u["question_id"] not in routed]
if XT_LIMIT:
    gen_unseen = gen_unseen[:XT_LIMIT]
print("unseen: %d | routed by exact match: %d (%.1f%%) | to generate: %d" % (len(unseen), len(routed), 100 * len(routed) / len(unseen), len(gen_unseen)))

test_rows = [r for r in rows if r["split"] == "test"]
hold = random.Random(42).sample(test_rows, min(N_HOLDOUT, len(test_rows))) if N_HOLDOUT else test_rows
print("hold-out rows (never seen in training):", len(hold), "of", len(test_rows))

# (name, adapter, uid tag, prompt mode, neighbour lists)
SYSTEMS = [("B1 zero-shot base model", "base", "hold", "none", None),
           ("B2 fine-tuned v1, no retrieval", "none_v1", "hold", "none", None),
           ("P1 fine-tuned v1 + retrieval (Phase 4)", "nn_v1", "hold", "nn", NNO),
           ("P2 FINAL fine-tuned v2 + re-ranked retrieval", "nn_v2", "hold", "nn", NNR)]
if ABLATION_ORIG_ORDER:
    SYSTEMS.append(("P2 ablation: original retrieval order", "nn_v2", "hold_orig", "nn", NNO))
FINAL = "P2 FINAL fine-tuned v2 + re-ranked retrieval"

def mk(adapter, prio, tag, qid, question, mode, NNs, split):
    ex = P.make_example(ctx, mode, qid, question, NNs[split].get(qid, []) if mode != "none" else [], tok=tok)
    return {"uid": "%s|%s|%s" % (adapter, tag, qid), "adapter": adapter, "prio": prio, "system": ex["system"], "user": ex["user"]}

jobs = []
for r in hold:
    for name, ad, tag, mode, NNs in SYSTEMS:
        jobs.append(mk(ad, 0, tag, r["question_id"], r["question"], mode, NNs, "test"))
for u in gen_unseen:
    jobs.append(mk("nn_v2", 1, "xt", u["question_id"], u["question"], "nn", NNR, "xt"))
os.makedirs(WORK + "/gen", exist_ok=True)
with open(WORK + "/jobs.jsonl", "w", encoding="utf-8") as f:
    for j in jobs:
        f.write(json.dumps(j, ensure_ascii=False) + "\n")
print("jobs:", len(jobs), dict(C.Counter(j["adapter"] for j in jobs)))

## 2. Generate - one worker per GPU (resumable)

In [ ]:
if not CPU:
    from huggingface_hub import snapshot_download
    snapshot_download(MODEL)
workers = [["nn_v2"], ["base", "none_v1", "nn_v1"]] if N_GPU >= 2 else [["base", "none_v1", "nn_v1", "nn_v2"]]
procs = []
for gpu, names in enumerate(workers):
    log = open("%s/gen/worker%d.log" % (WORK, gpu), "a")
    cmd = [sys.executable, "infer.py", "--jobs", WORK + "/jobs.jsonl", "--adapter_dir", AD, "--adapters", ",".join(names),
           "--out", "%s/gen/gen_gpu%d.jsonl" % (WORK, gpu), "--model", MODEL, "--batch", str(BATCH), "--hours", str(GEN_HOURS),
           "--max_new", str(DECODE["max_new"]), "--rep_penalty", str(DECODE["rep_penalty"]),
           "--gen_rep_penalty", str(DECODE["gen_rep_penalty"]), "--gen_no_repeat", str(DECODE["gen_no_repeat"])]
    if CPU:
        cmd.append("--no_quant")
    env = dict(os.environ, TOKENIZERS_PARALLELISM="false")
    if not CPU:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    procs.append((gpu, subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT)))
    print("started worker", gpu, names)
t0 = time.time()
while any(p.poll() is None for _, p in procs):
    time.sleep(120 if not CPU else 3)
    for gpu, p in procs:
        lines = open("%s/gen/worker%d.log" % (WORK, gpu), errors="ignore").read().splitlines()
        print("[%4.0f min] worker %d: %s" % ((time.time() - t0) / 60, gpu, lines[-1][:150] if lines else "(starting)"), flush=True)
for gpu, p in procs:
    print("worker", gpu, "exit code", p.returncode)
    if p.returncode:
        print(open("%s/gen/worker%d.log" % (WORK, gpu), errors="ignore").read()[-3000:])

## 3. Collect the answers

In [ ]:
import re
gen = {}
for f in glob.glob(WORK + "/gen/gen_gpu*.jsonl"):
    for l in open(f, encoding="utf-8"):
        if l.strip():
            d = json.loads(l); gen[d["uid"]] = d
missing = [j["uid"] for j in jobs if j["uid"] not in gen]
print("generated: %d of %d jobs | missing: %d" % (len(gen), len(jobs), len(missing)))
if missing:
    print("WARNING: jobs missing (a worker stopped early) - run the notebook again; it resumes. First:", missing[:3])
WORD = re.compile(r"\S+")
def is_loop(text, n=8, min_rep=3):
    w = [x.lower().strip(".,;:*#-()") for x in WORD.findall(text)]
    c = C.Counter(tuple(w[i:i + n]) for i in range(len(w) - n + 1))
    return bool(c) and max(c.values()) >= min_rep

## 4. Hold-out evaluation - the main results table

In [ ]:
import numpy as np, pandas as pd
import eval_harness as H
def nearest(NNs, split, q):
    nb = NNs[split].get(q)
    return qid2row[nb[0][0]]["final_answer"] if nb else ""

ids = [r["question_id"] for r in hold if all(("%s|%s|%s" % (ad, tag, r["question_id"])) in gen for _, ad, tag, _, _ in SYSTEMS)]
refs = {r["question_id"]: r["final_answer"] for r in hold if r["question_id"] in set(ids)}
systems = {"B0a copy nearest answer (original order)": {q: nearest(NNO, "test", q) for q in ids},
           "B0a copy nearest answer (re-ranked)": {q: nearest(NNR, "test", q) for q in ids}}
extra = {n: {} for n in systems}
for name, ad, tag, mode, NNs in SYSTEMS:
    p, e, cap = {}, 0, 0
    for q in ids:
        d = gen["%s|%s|%s" % (ad, tag, q)]
        cap += d["hit_limit"]
        if d["text"].strip():
            p[q] = d["text"]
        else:
            p[q], e = nearest(NNR, "test", q), e + 1     # empty generation -> nearest answer (counted)
    systems[name] = p
    extra[name] = {"empty_replaced": e, "hit_cap": cap}
res = {}
for n, p in systems.items():
    res[n] = H.evaluate(p, refs, bertscore=BERTSCORE)
    res[n].update(extra[n])
    res[n]["loops_8gram_x3"] = sum(is_loop(p[q]) for q in ids)
    res[n]["median_words"] = float(np.median([len(p[q].split()) for q in ids]))
cols = ["rougeL", "rouge1", "rouge2", "bleu", "fact_f1", "len_ratio_mean", "median_words", "hit_cap", "loops_8gram_x3", "empty_replaced"] + (["bertscore_f1"] if BERTSCORE else [])
table = pd.DataFrame({n: {c: r.get(c, np.nan) for c in cols} for n, r in res.items()}).T
pd.set_option("display.width", 250)
print("hold-out questions scored: %d (reference median %d words)" % (len(ids), np.median([len(refs[q].split()) for q in ids])))
print(table.round(4).to_string())

from rouge_score import rouge_scorer
sc = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
per = {n: np.array([sc.score(refs[q], p[q])["rougeL"].fmeasure for q in ids]) for n, p in systems.items()}
idx = np.random.default_rng(0).integers(0, len(ids), (2000, len(ids)))
def diff_ci(a, b):
    d = (per[a] - per[b])[idx].mean(axis=1)
    return "%+.4f  [95%% CI %+.4f, %+.4f]" % (per[a].mean() - per[b].mean(), np.percentile(d, 2.5), np.percentile(d, 97.5))
B0O, B0R = "B0a copy nearest answer (original order)", "B0a copy nearest answer (re-ranked)"
pairs = [(FINAL, B0R, "final - copy re-ranked nearest answer (value of generating)"),
         (FINAL, B0O, "final - copy nearest answer, original order"),
         (FINAL, "P1 fine-tuned v1 + retrieval (Phase 4)", "final - Phase 4 model (value of retraining + re-ranking)"),
         (FINAL, "B2 fine-tuned v1, no retrieval", "final - no retrieval"),
         (FINAL, "B1 zero-shot base model", "final - zero-shot"),
         ("P1 fine-tuned v1 + retrieval (Phase 4)", B0O, "Phase 4 model - copy nearest (original order)"),
         (B0R, B0O, "re-ranked copy - original copy (value of the re-ranker alone)")]
if ABLATION_ORIG_ORDER:
    pairs.append((FINAL, "P2 ablation: original retrieval order", "final - same model with original order (re-ranker at inference)"))
boot = {}
print("\nROUGE-L paired bootstrap (n=%d, 2000 resamples):" % len(ids))
for a, b, label in pairs:
    boot[label] = diff_ci(a, b)
    print("  %-70s %s" % (label, boot[label]))

## 5. The router, scored separately
Only repeated questions can be routed, so this is measured on training questions that appear more than once: the stored answer is compared with a *different* human answer to the same question.

In [ ]:
groups = C.defaultdict(list)
for r in rows:
    groups[r["group_id"]].append(r)
rep = [g for g in groups.values() if len(g) >= 2]
rr = random.Random(7)
held, p_med, p_rand = {}, {}, {}
for g in rr.sample(rep, min(1500, len(rep))):
    med = next((x for x in g if x["is_medoid"] == "1"), g[0])
    h = rr.choice([x for x in g if x is not med])
    held[h["question_id"]] = h["final_answer"]
    p_med[h["question_id"]] = med["final_answer"]
    p_rand[h["question_id"]] = rr.choice([x for x in g if x is not h])["final_answer"]
router_res, rand_res = H.evaluate(p_med, held), H.evaluate(p_rand, held)
print("repeated-question groups: %d | scored: %d" % (len(rep), len(held)))
print("router (stored medoid answer): rougeL %.4f | random other human answer: rougeL %.4f (conservative)" % (router_res["rougeL"], rand_res["rougeL"]))
share = len(routed) / len(unseen)
print("\nESTIMATE for the unseen set (NOT a measurement: mixes a training-set router check with the hold-out score):")
for k in ("rougeL", "rouge1", "fact_f1"):
    print("   %-8s %.4f  (%.1f%% routed x router + %.1f%% generated x final model)" % (k, share * rand_res[k] + (1 - share) * res[FINAL][k], 100 * share, 100 * (1 - share)))

## 6. Write the predictions for the unseen set

In [ ]:
final, source, baseline, fallback = {}, {}, {}, 0
for u in unseen:
    q = u["question_id"]
    if q in routed:
        final[q] = baseline[q] = routed[q]; source[q] = "router"
        continue
    baseline[q] = nearest(NNR, "xt", q)       # baseline system: same router, then copy the re-ranked nearest answer
    d = gen.get("nn_v2|xt|%s" % q)
    if d and d["text"].strip():
        final[q], source[q] = d["text"], "generated"
    else:
        final[q], source[q] = baseline[q], "fallback_nearest"
        fallback += 1
assert len(final) == len(unseen) == 1000 and all(v.strip() for v in final.values()), "predictions incomplete"
def write(path, header, rowf):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(header)
        for u in unseen:
            w.writerow(rowf(u))
write(WORK + "/xt_predictions.csv", ["question_id", "question", "final_answer"], lambda u: [u["question_id"], u["question"], final[u["question_id"]]])
write(WORK + "/xt_predictions_detailed.csv", ["question_id", "source", "final_answer"], lambda u: [u["question_id"], source[u["question_id"]], final[u["question_id"]]])
write(WORK + "/xt_predictions_with_baseline.csv", ["question_id", "question", "final_answer", "baseline_answer"],
      lambda u: [u["question_id"], u["question"], final[u["question_id"]], baseline[u["question_id"]]])
print("wrote xt_predictions*.csv:", len(final), "rows |", dict(C.Counter(source.values())))
if fallback:
    print("WARNING: %d unseen questions used the nearest-answer fallback (missing generations) - re-run to resume." % fallback)
xt_gen = [gen[u] for u in gen if u.startswith("nn_v2|xt|")]
json.dump({"holdout_n": len(ids), "decoding": DECODE, "systems": res, "bootstrap_rougeL": boot,
           "router": router_res, "router_random_other": rand_res, "routed_share": share,
           "unseen": {"sources": dict(C.Counter(source.values())), "generated_hit_cap": sum(d["hit_limit"] for d in xt_gen),
                      "generated_loops_8gram_x3": sum(is_loop(d["text"]) for d in xt_gen)},
           "retrained_adapter_training": {k: v for k, v in V2_STATUS.items() if k not in ("loss_log", "eval_log")},
           "config": {"MODEL": MODEL, "N_HOLDOUT": N_HOLDOUT, "ABLATION_ORIG_ORDER": ABLATION_ORIG_ORDER}},
          open(WORK + "/metrics.json", "w"), indent=1, default=float)
print("wrote metrics.json")

## 7. A few answers side by side (for the report's qualitative section)

In [ ]:
for q in ids[:3]:
    print("=" * 100, "\nQ:", qid2row[q]["question"][:200])
    print("--- reference :", refs[q][:400].replace("\n", " "))
    print("--- final (P2):", systems[FINAL][q][:400].replace("\n", " "))
    print("--- B0a re-ranked copy:", systems[B0R][q][:250].replace("\n", " "))

## After it finishes
Send me the notebook link (I download the output with the Kaggle CLI). Output tab: `xt_predictions.csv` (the deliverable),
`xt_predictions_with_baseline.csv`, `xt_predictions_detailed.csv`, `metrics.json`, `gen/` (every raw answer), `jobs.jsonl` (every prompt).